# BaSiC Caching Validation Test

**Objective**: Determine if computing BaSiC correction once (cached) produces equivalent quality to computing it individually per z-plane.

## Test Cases
1. **DAPI (CH1)** - High signal, present in all cycles
2. **Blank channels (Cycle 1 CH2-4)** - Background/noise only
3. **Sparse marker (Cycle 2 CH3)** - Fewer positive cells
4. **Dense marker (Cycle 3 CH3)** - More positive cells

## Processing Modes
- **Mode A (Individual)**: Compute BaSiC per z-plane
- **Mode B (Cached)**: Compute BaSiC once from reference plane, apply to all z-planes

## Metrics
- Intensity statistics (mean, std, CV)
- Flatfield uniformity
- Corrected image quality
- Visual comparison

In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib widget

In [2]:
import os
import sys
import numpy as np
import pandas as pd
from glob import glob
from datetime import datetime
import matplotlib.pyplot as plt
from scipy import stats
from skimage.io import imread
from skimage.io.collection import alphanumeric_key
import warnings
warnings.filterwarnings('ignore')

# Add CUDA bin to PATH for NVRTC DLLs (must be done before importing cupy)
cuda_bin = r"C:\Program Files\NVIDIA GPU Computing Toolkit\CUDA\v12.1\bin"
if os.path.exists(cuda_bin) and cuda_bin not in os.environ.get('PATH', ''):
    os.environ['PATH'] = cuda_bin + os.pathsep + os.environ.get('PATH', '')
    print(f"Added CUDA bin to PATH: {cuda_bin}")

# Add src to path
base_dir = "C:\\Users\\smith6jt"
sys.path.insert(0, os.path.join(base_dir, 'KINTSUGI', 'src'))

from kintsugi.kcorrect_gpu import KCorrectGPU, KCorrectGPUFunc, check_gpu

# Check GPU
gpu_available, gpu_msg = check_gpu()
print(f"GPU: {gpu_msg}")
print(f"Test started: {datetime.now()}")

Added CUDA bin to PATH: C:\Program Files\NVIDIA GPU Computing Toolkit\CUDA\v12.1\bin
GPU: GPU available: Device 0 with 21.5 GB
Test started: 2025-12-10 18:20:42.391777


In [3]:
# Configuration
data_dir = os.path.join(base_dir, 'KINTSUGI', 'data', '2008CC2B_raw')

# Test cases: (cycle, channel, name, description)
TEST_CASES = [
    (1, 1, "DAPI", "High signal reference"),
    (1, 2, "Blank_C1_CH2", "Blank control"),
    (1, 3, "Blank_C1_CH3", "Blank control"),
    (1, 4, "Blank_C1_CH4", "Blank control"),
    (2, 2, "Sparse_Marker_C2_CH3", "CD31 - sparse marker"),
    (2, 3, "Dense_Marker_C2_CH2", "CD8 - denser marker"),
]

# Check which cycles exist
available_cycles = [int(d[3:]) for d in os.listdir(data_dir) if d.startswith('cyc')]
print(f"Available cycles: {sorted(available_cycles)}")

# Filter test cases to available cycles
TEST_CASES = [(c, ch, n, d) for c, ch, n, d in TEST_CASES if c in available_cycles]
print(f"\nTest cases to run: {len(TEST_CASES)}")
for c, ch, n, d in TEST_CASES:
    print(f"  Cycle {c} CH{ch}: {n} ({d})")

# Parameters
N_ZPLANES = 17
REF_ZPLANE = N_ZPLANES // 2  # Middle z-plane as reference
TEST_ZPLANES = [1, 5, REF_ZPLANE, 12, 17]  # Sample z-planes to compare

print(f"\nReference z-plane: Z{REF_ZPLANE:02d}")
print(f"Test z-planes: {TEST_ZPLANES}")

Available cycles: [1, 2, 3, 13]

Test cases to run: 6
  Cycle 1 CH1: DAPI (High signal reference)
  Cycle 1 CH2: Blank_C1_CH2 (Blank control)
  Cycle 1 CH3: Blank_C1_CH3 (Blank control)
  Cycle 1 CH4: Blank_C1_CH4 (Blank control)
  Cycle 2 CH2: Sparse_Marker_C2_CH3 (CD31 - sparse marker)
  Cycle 2 CH3: Dense_Marker_C2_CH2 (CD8 - denser marker)

Reference z-plane: Z08
Test z-planes: [1, 5, 8, 12, 17]


In [ ]:
def load_tiles_for_zplane(data_dir, cycle, channel, zplane):
    """Load all tiles for a specific z-plane."""
    pattern = f'1_000??_Z0{str(zplane).zfill(2)}_CH{str(channel)}.tif'
    files = sorted(glob(os.path.join(data_dir, f'cyc{str(cycle).zfill(3)}', pattern)), 
                   key=alphanumeric_key)
    if not files:
        return None
    tiles = np.stack([imread(f) for f in files], axis=0)
    return tiles


def compute_metrics(corrected, flatfield, darkfield, original):
    """Compute quality metrics for corrected images."""
    metrics = {}
    
    # Intensity statistics
    metrics['mean'] = float(np.mean(corrected))
    metrics['std'] = float(np.std(corrected))
    metrics['min'] = float(np.min(corrected))
    metrics['max'] = float(np.max(corrected))
    metrics['cv'] = metrics['std'] / (metrics['mean'] + 1e-10)  # Coefficient of variation
    
    # Flatfield quality
    metrics['flatfield_mean'] = float(np.mean(flatfield))
    metrics['flatfield_std'] = float(np.std(flatfield))
    metrics['flatfield_uniformity'] = metrics['flatfield_std'] / (metrics['flatfield_mean'] + 1e-10)
    
    # Flatfield center-to-edge ratio
    h, w = flatfield.shape
    center = flatfield[h//4:3*h//4, w//4:3*w//4]
    edge = np.concatenate([
        flatfield[:h//8, :].flatten(),
        flatfield[-h//8:, :].flatten(),
        flatfield[:, :w//8].flatten(),
        flatfield[:, -w//8:].flatten()
    ])
    metrics['flatfield_center_edge_ratio'] = float(np.mean(center) / (np.mean(edge) + 1e-10))
    
    # Darkfield
    metrics['darkfield_mean'] = float(np.mean(darkfield))
    metrics['darkfield_std'] = float(np.std(darkfield))
    
    # Inter-tile variation (CV across tile means)
    tile_means = [np.mean(corrected[i]) for i in range(corrected.shape[0])]
    metrics['inter_tile_cv'] = float(np.std(tile_means) / (np.mean(tile_means) + 1e-10))
    
    # SNR estimate (signal / background noise)
    # Use median as signal estimate and MAD as noise
    median_val = np.median(corrected)
    mad = np.median(np.abs(corrected - median_val))
    metrics['snr'] = float(median_val / (1.4826 * mad + 1e-10))  # 1.4826 converts MAD to std
    
    return metrics


def compare_metrics(metrics_a, metrics_b, name_a='Individual', name_b='Cached'):
    """Compare two sets of metrics and compute percent differences."""
    comparison = {}
    for key in metrics_a:
        val_a = metrics_a[key]
        val_b = metrics_b[key]
        if abs(val_a) > 1e-10:
            pct_diff = 100 * (val_b - val_a) / val_a
        else:
            pct_diff = 0 if abs(val_b) < 1e-10 else float('inf')
        comparison[key] = {
            name_a: val_a,
            name_b: val_b,
            'pct_diff': pct_diff
        }
    return comparison

---
## Run Validation Test

In [ ]:
# Store all results
all_results = []

# BaSiC parameters
basic_params = {
    'if_darkfield': True,
    'max_iterations': 500,
    'optimization_tolerance': 1e-6,
    'max_reweight_iterations': 25,
    'reweight_tolerance': 1e-3,
    'use_gpu': True  # Use GPU for speed
}

for cycle, channel, name, description in TEST_CASES:
    print(f"\n{'='*70}")
    print(f"Testing: {name} (Cycle {cycle} CH{channel}) - {description}")
    print(f"{'='*70}")
    
    case_results = {
        'name': name,
        'cycle': cycle,
        'channel': channel,
        'description': description,
        'zplane_comparisons': []
    }
    
    # Step 1: Compute cached flatfield/darkfield from reference z-plane
    print(f"\n[1] Computing CACHED correction from reference Z{REF_ZPLANE:02d}...")
    ref_tiles = load_tiles_for_zplane(data_dir, cycle, channel, REF_ZPLANE)
    if ref_tiles is None:
        print(f"  ERROR: Could not load reference tiles")
        continue
    
    # Normalize to 0-1 range
    dtype_max = np.iinfo(ref_tiles.dtype).max
    ref_tiles_norm = ref_tiles.astype(np.float64) / dtype_max
    
    cached_start = datetime.now()
    flatfield_cached, darkfield_cached = KCorrectGPUFunc(ref_tiles_norm, **basic_params)
    cached_time = (datetime.now() - cached_start).total_seconds()
    print(f"  Cached correction computed in {cached_time:.1f}s")
    print(f"  Flatfield range: [{flatfield_cached.min():.3f}, {flatfield_cached.max():.3f}]")
    print(f"  Darkfield range: [{darkfield_cached.min():.3f}, {darkfield_cached.max():.3f}]")
    
    case_results['cached_flatfield'] = flatfield_cached.copy()
    case_results['cached_darkfield'] = darkfield_cached.copy()
    case_results['cached_compute_time'] = cached_time
    
    # Step 2: Test each z-plane
    print(f"\n[2] Comparing INDIVIDUAL vs CACHED for z-planes: {TEST_ZPLANES}")
    
    for z in TEST_ZPLANES:
        print(f"\n  --- Z{z:02d} ---")
        
        # Load tiles for this z-plane
        tiles = load_tiles_for_zplane(data_dir, cycle, channel, z)
        if tiles is None:
            print(f"    Could not load tiles for Z{z:02d}")
            continue
        tiles_norm = tiles.astype(np.float64) / dtype_max
        
        # Mode A: Individual BaSiC for this z-plane
        print(f"    Computing INDIVIDUAL BaSiC...")
        indiv_start = datetime.now()
        flatfield_indiv, darkfield_indiv = KCorrectGPUFunc(tiles_norm, **basic_params)
        indiv_time = (datetime.now() - indiv_start).total_seconds()
        print(f"    Individual: {indiv_time:.1f}s")
        
        # Apply individual correction
        corrected_indiv = (tiles_norm - darkfield_indiv) / (flatfield_indiv + 1e-10)
        corrected_indiv = np.clip(corrected_indiv, 0, 1)
        
        # Mode B: Apply cached correction
        corrected_cached = (tiles_norm - darkfield_cached) / (flatfield_cached + 1e-10)
        corrected_cached = np.clip(corrected_cached, 0, 1)
        
        # Compute metrics for both
        metrics_indiv = compute_metrics(corrected_indiv, flatfield_indiv, darkfield_indiv, tiles_norm)
        metrics_cached = compute_metrics(corrected_cached, flatfield_cached, darkfield_cached, tiles_norm)
        
        # Compare
        comparison = compare_metrics(metrics_indiv, metrics_cached)
        
        # Store results
        zplane_result = {
            'zplane': z,
            'indiv_time': indiv_time,
            'metrics_indiv': metrics_indiv,
            'metrics_cached': metrics_cached,
            'comparison': comparison,
            'flatfield_indiv': flatfield_indiv.copy(),
            'darkfield_indiv': darkfield_indiv.copy()
        }
        case_results['zplane_comparisons'].append(zplane_result)
        
        # Print key differences
        print(f"    Key metric differences (Individual vs Cached):")
        for key in ['mean', 'inter_tile_cv', 'snr', 'flatfield_uniformity']:
            c = comparison[key]
            print(f"      {key}: {c['Individual']:.4f} vs {c['Cached']:.4f} ({c['pct_diff']:+.2f}%)")
    
    all_results.append(case_results)

print(f"\n\n{'='*70}")
print("Test completed!")
print(f"{'='*70}")

---
## Analysis and Summary

In [ ]:
# Build summary table
summary_rows = []

for case in all_results:
    for zcomp in case['zplane_comparisons']:
        row = {
            'Test Case': case['name'],
            'Cycle': case['cycle'],
            'Channel': case['channel'],
            'Z-plane': zcomp['zplane'],
            'Individual Time (s)': zcomp['indiv_time'],
        }
        # Add key metrics and their percent differences
        for metric in ['mean', 'inter_tile_cv', 'snr', 'flatfield_uniformity']:
            c = zcomp['comparison'][metric]
            row[f'{metric}_indiv'] = c['Individual']
            row[f'{metric}_cached'] = c['Cached']
            row[f'{metric}_pct_diff'] = c['pct_diff']
        summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)

# Display summary
print("\n" + "="*80)
print("SUMMARY TABLE")
print("="*80)

# Show key columns
display_cols = ['Test Case', 'Z-plane', 'mean_pct_diff', 'inter_tile_cv_pct_diff', 
                'snr_pct_diff', 'flatfield_uniformity_pct_diff']
print(summary_df[display_cols].to_string(index=False))

In [ ]:
# Evaluate pass/fail criteria
PASS_THRESHOLD = 5.0  # Pass if all metrics within 5%
WARN_THRESHOLD = 10.0  # Warning if metrics between 5-10%

print("\n" + "="*80)
print("PASS/FAIL EVALUATION")
print(f"Pass threshold: <{PASS_THRESHOLD}% difference")
print(f"Warning threshold: {PASS_THRESHOLD}-{WARN_THRESHOLD}% difference")
print(f"Fail threshold: >{WARN_THRESHOLD}% difference")
print("="*80 + "\n")

metrics_to_check = ['mean', 'inter_tile_cv', 'snr', 'flatfield_uniformity']

overall_pass = True
overall_warnings = []
overall_failures = []

for case in all_results:
    print(f"\n{case['name']} (Cycle {case['cycle']} CH{case['channel']}):")
    
    for zcomp in case['zplane_comparisons']:
        z = zcomp['zplane']
        status = "PASS"
        issues = []
        
        for metric in metrics_to_check:
            pct_diff = abs(zcomp['comparison'][metric]['pct_diff'])
            
            if pct_diff > WARN_THRESHOLD:
                status = "FAIL"
                issues.append(f"{metric}={pct_diff:.1f}%")
                overall_failures.append((case['name'], z, metric, pct_diff))
            elif pct_diff > PASS_THRESHOLD:
                if status != "FAIL":
                    status = "WARN"
                issues.append(f"{metric}={pct_diff:.1f}%")
                overall_warnings.append((case['name'], z, metric, pct_diff))
        
        if status == "FAIL":
            overall_pass = False
        
        status_symbol = {"PASS": "OK", "WARN": "WARNING", "FAIL": "FAIL"}[status]
        issue_str = f" [{', '.join(issues)}]" if issues else ""
        print(f"  Z{z:02d}: {status_symbol}{issue_str}")

print("\n" + "="*80)
if overall_pass and not overall_warnings:
    print("OVERALL RESULT: PASS - Caching produces equivalent results")
    print("Recommendation: SAFE to implement caching for speed improvement")
elif overall_pass:
    print(f"OVERALL RESULT: PASS with {len(overall_warnings)} warnings")
    print("Recommendation: Caching acceptable, but review warnings")
else:
    print(f"OVERALL RESULT: FAIL - {len(overall_failures)} failures detected")
    print("Recommendation: DO NOT implement caching - individual z-plane processing required")
    print("\nFailures:")
    for name, z, metric, diff in overall_failures:
        print(f"  {name} Z{z:02d}: {metric} differs by {diff:.1f}%")
print("="*80)

---
## Visual Comparison

In [ ]:
# Compare flatfields across z-planes
for case in all_results:
    print(f"\n{case['name']}:")
    
    n_z = len(case['zplane_comparisons'])
    fig, axes = plt.subplots(2, n_z + 1, figsize=(3*(n_z+1), 6))
    fig.suptitle(f"Flatfield Comparison: {case['name']}", fontsize=14)
    
    # Cached flatfield (first column)
    axes[0, 0].imshow(case['cached_flatfield'], cmap='viridis')
    axes[0, 0].set_title('Cached\n(Reference)')
    axes[0, 0].axis('off')
    
    axes[1, 0].imshow(case['cached_darkfield'], cmap='viridis')
    axes[1, 0].set_title('Cached Darkfield')
    axes[1, 0].axis('off')
    
    # Individual flatfields per z-plane
    for i, zcomp in enumerate(case['zplane_comparisons']):
        col = i + 1
        
        axes[0, col].imshow(zcomp['flatfield_indiv'], cmap='viridis')
        axes[0, col].set_title(f"Z{zcomp['zplane']:02d}\nIndividual")
        axes[0, col].axis('off')
        
        axes[1, col].imshow(zcomp['darkfield_indiv'], cmap='viridis')
        axes[1, col].set_title(f"Z{zcomp['zplane']:02d} Darkfield")
        axes[1, col].axis('off')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Plot percent differences across z-planes
metrics_to_plot = ['mean', 'inter_tile_cv', 'snr']

fig, axes = plt.subplots(len(all_results), len(metrics_to_plot), 
                         figsize=(4*len(metrics_to_plot), 3*len(all_results)))
if len(all_results) == 1:
    axes = axes.reshape(1, -1)

for row, case in enumerate(all_results):
    zplanes = [zc['zplane'] for zc in case['zplane_comparisons']]
    
    for col, metric in enumerate(metrics_to_plot):
        diffs = [zc['comparison'][metric]['pct_diff'] for zc in case['zplane_comparisons']]
        
        ax = axes[row, col]
        colors = ['green' if abs(d) < 5 else 'orange' if abs(d) < 10 else 'red' for d in diffs]
        ax.bar(range(len(zplanes)), diffs, color=colors)
        ax.set_xticks(range(len(zplanes)))
        ax.set_xticklabels([f'Z{z}' for z in zplanes])
        ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
        ax.axhline(y=5, color='orange', linestyle='--', linewidth=0.5)
        ax.axhline(y=-5, color='orange', linestyle='--', linewidth=0.5)
        ax.axhline(y=10, color='red', linestyle='--', linewidth=0.5)
        ax.axhline(y=-10, color='red', linestyle='--', linewidth=0.5)
        ax.set_ylabel('% Difference')
        
        if row == 0:
            ax.set_title(metric)
        if col == 0:
            ax.set_ylabel(f"{case['name']}\n% Difference")

plt.tight_layout()
plt.suptitle('Cached vs Individual: % Difference by Z-plane', y=1.02, fontsize=14)
plt.show()

---
## Conclusions

Based on the test results above, determine:

1. **Is caching acceptable?** Check if all metrics are within the pass threshold.
2. **Which test cases show the largest differences?** Blank channels vs markers?
3. **Is there a pattern across z-planes?** Do edge z-planes (Z1, Z17) differ more than middle z-planes?

### Action Items

- If **PASS**: Implement caching for significant speed improvement
- If **WARN**: Review specific cases, may need selective caching
- If **FAIL**: Keep individual z-plane processing, optimize other areas

In [ ]:
# Save results to file
results_path = os.path.join(base_dir, 'KINTSUGI', 'docs', 'basic_caching_validation_results.csv')
summary_df.to_csv(results_path, index=False)
print(f"Results saved to: {results_path}")

print(f"\nTest completed at {datetime.now()}")